# Multi-Task Model Training

ResNet18의 백본을 공유하고 Head만 분기하는 Multi-Task 모델을 번갈아가며 학습(Alternating Training)합니다.

In [1]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import glob
import PIL.Image
import os

class MultiTaskResNet(nn.Module):
    def __init__(self):
        super(MultiTaskResNet, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.xy_head = nn.Linear(512, 2)
        self.finger_head = nn.Linear(512, 6)

    def forward(self, x, mode='both'):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        if mode == 'line':
            return self.xy_head(x)
        elif mode == 'finger':
            return self.finger_head(x)
        else:
            return self.xy_head(x), self.finger_head(x)


In [ ]:
DATASET_DIR = 'dataset_multi'
LINE_DIR = os.path.join(DATASET_DIR, 'line')
FINGER_DIR = os.path.join(DATASET_DIR, 'finger')
CLASSES = ['0', '1', '2', '3', '4', '5']

def get_x(path): return (float(int(path[3:6])) - 50.0) / 50.0
def get_y(path): return (float(int(path[7:10])) - 50.0) / 50.0

class XYDataset(torch.utils.data.Dataset):
    def __init__(self, directory):
        self.image_paths = glob.glob(os.path.join(directory, '*.jpg'))
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = PIL.Image.open(image_path)
        x = get_x(os.path.basename(image_path))
        y = get_y(os.path.basename(image_path))
        image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        image = torch.from_numpy(image.numpy()[::-1].copy())
        image = transforms.functional.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return image, torch.tensor([x, y]).float()

class FingerDataset(torch.utils.data.Dataset):
    def __init__(self, directory):
        self.image_paths = []
        self.labels = []
        for label in CLASSES:
            label_dir = os.path.join(directory, label)
            if os.path.isdir(label_dir):
                for img_name in os.listdir(label_dir):
                    if img_name.endswith('.jpg'):
                        self.image_paths.append(os.path.join(label_dir, img_name))
                        self.labels.append(int(label))
        self.color_jitter = transforms.ColorJitter(0.3, 0.3, 0.3, 0.3)
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image = PIL.Image.open(self.image_paths[idx])
        label = self.labels[idx]
        image = self.color_jitter(image)
        image = transforms.functional.resize(image, (224, 224))
        image = transforms.functional.to_tensor(image)
        image = torch.from_numpy(image.numpy()[::-1].copy())
        image = transforms.functional.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return image, label

line_dataset = XYDataset(LINE_DIR)
finger_dataset = FingerDataset(FINGER_DIR)

print(f"Line samples: {len(line_dataset)}, Finger samples: {len(finger_dataset)}")


In [ ]:
if len(line_dataset) > 0 and len(finger_dataset) > 0:
    n_test_line = max(1, int(0.1 * len(line_dataset)))
    train_line, test_line = torch.utils.data.random_split(line_dataset, [len(line_dataset) - n_test_line, n_test_line])

    n_test_finger = max(1, int(0.1 * len(finger_dataset)))
    train_finger, test_finger = torch.utils.data.random_split(finger_dataset, [len(finger_dataset) - n_test_finger, n_test_finger])

    line_train_loader = torch.utils.data.DataLoader(train_line, batch_size=16, shuffle=True)
    line_test_loader = torch.utils.data.DataLoader(test_line, batch_size=16, shuffle=True)

    finger_train_loader = torch.utils.data.DataLoader(train_finger, batch_size=16, shuffle=True)
    finger_test_loader = torch.utils.data.DataLoader(test_finger, batch_size=16, shuffle=True)
else:
    print("Warning: Ensure dataset_multi/line and dataset_multi/finger have images.")


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskResNet().to(device)
optimizer = optim.Adam(model.parameters())
NUM_EPOCHS = 10
BEST_MODEL_PATH = 'best_model_multi.pth'
best_loss = 1e9

for epoch in range(NUM_EPOCHS):
    model.train()
    
    # 1. Line Tracing Batch
    train_loss_line = 0.0
    for images, labels in line_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, mode='line')
        loss = F.mse_loss(outputs, labels)
        train_loss_line += float(loss)
        loss.backward()
        optimizer.step()
    train_loss_line = train_loss_line / len(line_train_loader) if len(line_train_loader) > 0 else 0
    
    # 2. Finger Classification Batch
    train_loss_finger = 0.0
    for images, labels in finger_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images, mode='finger')
        loss = F.cross_entropy(outputs, labels)
        train_loss_finger += float(loss)
        loss.backward()
        optimizer.step()
    train_loss_finger = train_loss_finger / len(finger_train_loader) if len(finger_train_loader) > 0 else 0
        
    print(f'Epoch {epoch}: Train(Line: {train_loss_line:.4f}, Finger: {train_loss_finger:.4f})')
    torch.save(model.state_dict(), BEST_MODEL_PATH)

print('Finished Training!')